#Initialization

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

#Read silver table

In [0]:
df = spark.table("pcat.silver.products")
df = df.select("product_code", "product_id", "division", "category", "product", "variant")

#Writing Gold Table

In [0]:
df.write \
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable("pcat.gold.sb_dim_products")

#Merging Data source with parent table

In [0]:
delta_table = DeltaTable.forName(spark, "pcat.gold.dim_products")
df_child_products = spark.table("pcat.gold.sb_dim_products").select(
    "product_code",
    "division",
    "category",
    "product",
    "variant"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()